# 第12章　フェデレーテッドラーニング ― データを動かさずに学ぶ

**『医療診断支援AIの社会実装（社会実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-social

## FedAvgの一巡を、手順で追う

In [ ]:
# 連合学習1ラウンド（中央サーバー側）の骨格
global_w = init_weights()
for rnd in range(num_rounds):
    updates, sizes = [], []
    for hospital in participants:              # 各施設へ並列配布
        w_k = hospital.local_train(            # 施設内でEエポック学習
            weights=copy(global_w), epochs=E, lr=lr)
        updates.append(w_k)
        sizes.append(hospital.n_samples)       # 送るのは件数と重みだけ
    n = sum(sizes)
    global_w = sum((nk / n) * wk               # FedAvg = データ量で加重平均
                   for wk, nk in zip(updates, sizes))

## 一つの世界モデルでは足りない ― パーソナライゼーションと階層FL

In [ ]:
for hospital in participants:
    w_body = copy(global_body)          # 共有する本体（body）だけ受け取る
    head_k = hospital.local_head        # ヘッドは各施設が保持し続ける
    w_body, head_k = hospital.train(w_body, head_k, epochs=E)
    body_updates.append((w_body, hospital.n))   # 送り返すのは body だけ
global_body = fedavg(body_updates)      # ヘッドは集約しない（施設固有のまま）